# Curate PDB assemblies from CIF 
2023-1-11

This notebook curates datasets of assemblies that may contain multiple protein chains and multiple ligand chains. These are intended to be used in phase 3 of training of the fold & dock 3 model.

The code below does:

1. Data curation, consisting of the following steps:

 - Ivan's CIFUtils parser is used to process all PDB entries downloaded in mmCIF format in Dec 2022. (During training, only examples from before 2020-04-30 are used)
 - Make a list of all small-molecule ligands, defined as sets of tuples (chain ID, residue number) with chain type "nonpoly" and containing covalent bonds to each other. Each of these "query" ligands will be the basis of a training example. Also identify if the ligand has covalent bonds to a protein chain.
 - Remove a manually chosen set of non-biological ligands (based on residue name)
 - Remove PDB ID / ligand name combinations that are not listed in PDBBind/BioLiP, to avoid low quality training examples. This removes 60-75% of ligands, so may be worth revisiting to see if some of the excluded examples are worth including in training.
 - Associate each query ligand with nearby protein and ligand chains. The protein chain with the most contacts is the "main" chain for this training example and its sequence is used for various filtering and merging operations later.
 - Exclude examples with protein chain similarity to Astex and CASF2016 examples, so we can use these sets for evaluation. We don't take ligand similarity into account.
 - Various filtering steps to avoid errors during featurization & training:
   - Remove covalent protein-ligand bonds that seem non-biological (O-O, F-F, -H2O, etc)
   - Remove examples where ligands are <1A from a protein atom
   - Remove examples where protein chains have a different number of residues from their pre-computed MSAs (these are mistakes during MSA curation).
   - Remove nucleic acid chains (these can be added back later, avoiding for now to simplify data loaders)
   - Label ligands with # of atoms. These are used to filter ligands larger than the crop size to control GPU memory usage
   
2. Dataset splitting. We make different mutually exclusive datasets with chemically/technically different ligands to control the proportions seen during training:

 - Protein / organic ligands (sm_compl)
 - Protein / metal ions (metal_compl)
 - Protein / multi-residue ligands (sm_compl_multi)
 - Protein / covalent ligands (sm_compl_covale)
 - Protein / ligand assemblies (more than 1 protein or ligand chain) (sm_compl_asmb)

3. Make a strict validation set for the protein / organic ligands where ligands with tanimoto similarity > 0.85 to the training set are removed.

## Imports

In [1]:
import sys, os, glob, pickle, gzip
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
sys.path.insert(0,'/home/jue/git/rf2a-fd3/')
sys.path.insert(0,'/home/jue/git/rf2a-fd3/rf2aa/')
import analysis_util
from parsers import *
from util import is_atom

/software/conda/envs/dlchem/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from curation_utils import *

## Pre-parse Ivan's dataset

In [11]:
sys.path.insert(0,'/home/jue/git/chemnet/arch.22-10-28/')
sys.path.insert(0,'/home/jue/git/chemnet/arch.22-10-28/pdb/')
import cifutils

In [9]:
Parser = cifutils.CIFParser()

CPU times: user 2.01 s, sys: 396 ms, total: 2.4 s
Wall time: 2.43 s


In [21]:
basedir = '/databases/rcsb/cif/'

In [22]:
subdirs =  glob.glob(basedir+'/*/')

In [23]:
len(subdirs)

1060

In [24]:
filenames = []

In [25]:
for subdir in subdirs:
    filenames += glob.glob(subdir+'/*.cif.gz')    

In [57]:
len(filenames)

199638

In [37]:
with open('all_rcsb_cif.txt','w') as outf:
    for fn in filenames:
        print(fn, file=outf)

In [38]:
filenames = [line.strip() for line in open('all_rcsb_cif.txt').readlines()]

In [41]:
for fn in filenames[istart:istart+num]:
    try:
        chains,asmb,covale,modres = Parser.parse(fn)
        outfn = fn.replace('/databases/rcsb/cif/','/projects/ml/RF2_allatom/rcsb/pkl/')
        outfn = outfn.replace('.cif.gz','.pkl.gz')
        outdir = os.path.dirname(outfn)
        os.makedirs(outdir, exist_ok=True)
        pickle.dump((chains,asmb,covale,modres),gzip.open(outfn,'wb'))
    except Exception as e:
        print(fn)
        print(repr(e))

Put the above code into a python script so it can be run in parallel on many nodes to process the full dataset

Generate a list of commands to run

In [58]:
num = 500

In [41]:
with open('parse_cifs_cmds.txt','w') as outf:
    for istart in range(0, len(filenames), num):
        print(f'source activate dlchem; python parse_cifs.py -istart {istart} -num {num}',
              file=outf)

In [44]:
filenames = glob.glob('/projects/ml/RF2_allatom/rcsb/pkl/*/*.pkl.gz')

In [45]:
len(filenames)

199638

## Get ligands
Use Ivan's parsed cif data structures to identify ligands ("nonpoly" residues) and save a list with information about them.

In [8]:
import time, pickle, gzip

In [9]:
sys.path.insert(0,'/home/jue/git/chemnet/arch.22-10-28/')
sys.path.insert(0,'/home/jue/git/chemnet/arch.22-10-28/pdb/')
import cifutils

In [10]:
cif_files = glob.glob('/projects/ml/RF2_allatom/rcsb/pkl/*/*.pkl.gz')

In [11]:
len(cif_files)

199638

In [12]:
from collections import Counter

In [37]:
records = []
ct = 0
start_time = time.time()

for fn in cif_files:

    pdbid = os.path.basename(fn).replace('.pkl.gz','')
    chains,asmb,covale,modres = pickle.load(gzip.open(fn))

    # collect all ligand residues and potential inter-ligand bonds
    lig_res_s = list(set([x[:3] for ch in chains.values() if ch.type=='nonpoly' 
                                for x in ch.atoms if x[2]!='HOH']))
    bonds = []
    for i_ch,ch in chains.items():
        if ch.type=='nonpoly':
            bonds.extend(ch.bonds)
    inter_ligand_bonds = []
    prot_lig_bonds = []
    for bond in covale:
        if chains[bond.a[0]].type=='nonpoly' and chains[bond.b[0]].type=='nonpoly':
            bonds.append(bond)
            inter_ligand_bonds.append(bond)
        if sorted([chains[bond.a[0]].type, chains[bond.b[0]].type])==['nonpoly','polypeptide(L)']:
            prot_lig_bonds.append(bond)

    # make list of bonded ligands (lists of ligand residues)
    ligands = []
    while len(lig_res_s)>0:
        res = lig_res_s[0]
        lig = get_bonded_partners(res, bonds)
        lig.add(res)
        lig = sorted(list(lig))
        lig_res_s = [res for res in lig_res_s if res not in lig]
        ligands.append(lig)

    for lig in ligands:
        records.append(dict(
            PDBID=pdbid, 
            LIGAND=lig,
            # this is incorrect -- may not contain bonds to the specific ligand `lig`
            # fixed this below before splitting into different datasets
            COVALENT=[(bond.a, bond.b) for bond in prot_lig_bonds],
        ))
        
    ct += 1
    if ct % 100 == 0:
        print(ct, time.time()-start_time)
    
    # if len(inter_ligand_bonds)>0: break
    if len(prot_lig_bonds)>0: break

In [38]:
df = pd.DataFrame.from_records(records)

Put the above code into a python script so it can be run in parallel on many nodes to process the full dataset

Generate a list of commands to run

In [43]:
num = 500

In [44]:
with open('cmds_get_ligands.txt','w') as outf:
    for istart in range(0, len(cif_files), num):
        print(f'source activate dlchem; python get_ligands.py -istart {istart} -num {num}',
              file=outf)

Compile the final list

In [45]:
len(cif_files)

199638

In [46]:
df = pd.concat([pd.read_csv(f'results_get_ligands/ligands{i}.csv',index_col=0) 
                for i in range(0,len(cif_files),num)])

In [47]:
df.shape

(1867647, 3)

In [57]:
df.to_csv('ligands.csv',index=None)

## Filter non-bio ligands

Remove these "non-biological" ligands. Previously curated by Gyu Rie and Linna, from `/home/gyurie/mpnn_ligand/bad_lig_list`. They were trying to exclude metal ions so I add these back, based on this list from the BioLiP website (extracted from the url when you click on their link for "PDB entries with metals")

Rohith and I added PO4, because these examples are numerous and poorly predicted

In [33]:
biolip_list = ['NUC','ZN','CA','MG','III','MN','FE','CU','SF4','FE2','CO','FES','GOL','NA',  
            'CL','K','CU1','GOL','XE','NO2','EDO','NI','BR','CD','O','CS','NO','TL','HG','UNL','KR',
            'SR','RB','F','AG','AR','U','AU','MO','SE','GD','YB','VX','SM','LI','RE','N','W','OS','HO','PI']
LA_list = ['EDO','PG4','OGA','SO4','HEZ','FEO','CL','DMS','ACT','MPD','GOL' ,'NH2','CUA','SIW','PGW','IOD','BR','3NI','ZRW','78M','UNX','nan']
rohith = ['MES','CCN','PO4']
metals = ['LA','NI','3CO','K','CR','ZN','CD','PD','TB','YT3','OS','EU','NA','RB','W','YB','HO3',
          'CE','MN','TL','LI','MN3','AU3','AU','EU3','AL','3NI','FE2','PT','FE','CA','AG','CU1',
          'LU','HG','CO','SR','MG','PB','CS','GA','BA','SM','SB','CU','MO','CU2']

In [34]:
exclude = set(biolip_list+LA_list+rohith)-set(metals)

In [35]:
df = pd.read_csv('ligands.csv')

In [36]:
df['LIGAND'] = df['LIGAND'].apply(lambda x: eval(x))

In [37]:
df.shape

(1867647, 3)

In [38]:
df_filt = df[~df['LIGAND'].apply(lambda x: x[0][2] in exclude)]

In [39]:
df_filt.shape

(1443023, 3)

In [40]:
df_filt.to_csv('ligands_filt.csv',index=None)

## Download PDBBind / BioLiP
2022-12-13

These are curated lists of biologically relevant ligands

### PDBBind

In [54]:
import re
from collections import OrderedDict

In [55]:
lig_re = re.compile('\((.*?)\)')

In [56]:
records = []
with open('/home/jue/dev/rfaa/index/INDEX_general_PL.2020') as f:
    lines = f.readlines()
    for line in lines:
        if not line.startswith('#'):
            tokens = line.split('//')
            desc = tokens[1]
            ligand = lig_re.findall(desc)[0]
            tokens = tokens[0].split()
            records.append(OrderedDict(
                pdb=tokens[0],
                resolution=tokens[1],
                year=tokens[2],
                affinity=tokens[3],
                ligand=ligand,
                description=desc
            ))

In [57]:
pdbbind = pd.DataFrame.from_records(records)

In [58]:
pdbbind.shape

(19443, 6)

In [112]:
pdbbind.to_csv('pdbbind_20221213.csv')

### BioLiP

In [80]:
import requests

In [96]:
for datestr in pd.date_range(start='2013-3-13', end='2022-03-30', freq='7D'):
    url = f'https://zhanggroup.org/BioLiP/weekly/BioLiP_{str(datestr).split()[0]}.txt'
    r = requests.get(url, allow_redirects=True)
    fn = os.path.basename(url)
    open(fn, 'wb').write(r.content)
    print(fn)

BioLiP_2013-03-13.txt
BioLiP_2013-03-20.txt
BioLiP_2013-03-27.txt
BioLiP_2013-04-03.txt
BioLiP_2013-04-10.txt
BioLiP_2013-04-17.txt
BioLiP_2013-04-24.txt
BioLiP_2013-05-01.txt
BioLiP_2013-05-08.txt
BioLiP_2013-05-15.txt
BioLiP_2013-05-22.txt
BioLiP_2013-05-29.txt
BioLiP_2013-06-05.txt
BioLiP_2013-06-12.txt
BioLiP_2013-06-19.txt
BioLiP_2013-06-26.txt
BioLiP_2013-07-03.txt
BioLiP_2013-07-10.txt
BioLiP_2013-07-17.txt
BioLiP_2013-07-24.txt
BioLiP_2013-07-31.txt
BioLiP_2013-08-07.txt
BioLiP_2013-08-14.txt
BioLiP_2013-08-21.txt
BioLiP_2013-08-28.txt
BioLiP_2013-09-04.txt
BioLiP_2013-09-11.txt
BioLiP_2013-09-18.txt
BioLiP_2013-09-25.txt
BioLiP_2013-10-02.txt
BioLiP_2013-10-09.txt
BioLiP_2013-10-16.txt
BioLiP_2013-10-23.txt
BioLiP_2013-10-30.txt
BioLiP_2013-11-06.txt
BioLiP_2013-11-13.txt
BioLiP_2013-11-20.txt
BioLiP_2013-11-27.txt
BioLiP_2013-12-04.txt
BioLiP_2013-12-11.txt
BioLiP_2013-12-18.txt
BioLiP_2013-12-25.txt
BioLiP_2014-01-01.txt
BioLiP_2014-01-08.txt
BioLiP_2014-01-15.txt
BioLiP_201

In [107]:
df_s = []
for datestr in pd.date_range(start='2013-3-06', end='2022-03-30', freq='7D'):
    fn = f'biolip/BioLiP_{str(datestr).split()[0]}.txt'
    
    dat = open(fn).read()
    if len(dat)==0 or '404 Not Found' in dat: continue
    
    tmp = pd.read_csv(fn,delimiter='\t',header=None)
    tmp.columns = ['pdb_id','pdb_chain','resolution','binding_site_id','ligand_id','ligand_chain','ligand_serial_number',
                  'bs_res','bs_res_renum','cat_site_res','cat_site_res','EC','GO','binding_affinity','binding_affinity_MOAD',
                  'binding_affinity_pdbbind','binding_affinity_bindingDB','uniprot','pubmed','receptor_seq']
    df_s.append(tmp)
biolip = pd.concat(df_s)

In [108]:
biolip.shape

(477226, 20)

In [111]:
biolip.to_csv('biolip_combined_20221213.csv')

## Filter to PDBBind / BioLiP

In [41]:
pdbbind = pd.read_csv('pdbbind_20221213.csv')
biolip = pd.read_csv('biolip_combined_20221213.csv')

/scratch/jue/26825327/ipykernel_386092/946346140.py:2: DtypeWarning: Columns (10,11,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  biolip = pd.read_csv('biolip_combined_20221213.csv')


In [42]:
biolip_entries = biolip[['pdb_id','ligand_id']].rename(columns={'pdb_id':'PDBID','ligand_id':'LIGNAME'})
biolip_entries['KEEP'] = True

In [43]:
pdbbind_entries = pdbbind[['pdb','ligand']].rename(columns={'pdb':'PDBID', 'ligand':'LIGNAME'}).drop_duplicates()
pdbbind_entries['KEEP'] = True

In [44]:
to_keep = biolip_entries.append(pdbbind_entries)
to_keep = to_keep.drop_duplicates()

/scratch/jue/26825327/ipykernel_386092/307790726.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  to_keep = biolip_entries.append(pdbbind_entries)


In [45]:
to_keep.shape

(166142, 3)

In [75]:
df = pd.read_csv('ligands_filt.csv')

In [ ]:
df['LIGAND'] = df['LIGAND'].apply(lambda x: eval(x))

In [47]:
df.shape

(1443023, 3)

In [48]:
records = []
for i,row in df.iterrows():
    for lig in row['LIGAND']:
        newrow = row.copy()
        newrow['LIGNAME'] = lig[2]
        records.append(newrow)

In [49]:
df2 = pd.DataFrame.from_records(records)

In [50]:
df2.shape

(1535977, 4)

In [51]:
tmp = df2.merge(to_keep, on=['PDBID','LIGNAME'], how='left')

In [52]:
tmp.shape

(1535977, 5)

In [53]:
df3 = tmp[tmp['KEEP']==True]

In [54]:
df3.shape

(395291, 5)

In [55]:
df3 = df3.drop(['LIGNAME','KEEP'], axis=1)

In [56]:
df3['LIGAND'] = df3['LIGAND'].apply(lambda x: str(x))

In [57]:
df3 = df3.drop_duplicates()

In [58]:
df3.shape

(382773, 3)

In [59]:
df3.to_csv('ligands_filt.csv',index=None)

## Get binding partners
Identify which protein and ligand chains are closest to each query ligand

### Run

In [339]:
df = pd.read_csv('ligands_filt.csv')

In [333]:
df['LIGAND'] = df['LIGAND'].apply(lambda x: eval(x))

In [334]:
df.shape

(382773, 3)

In [343]:
records = []
start_time = time.time()
pdbid_prev = None

for i,row in tmp.iterrows():
    pdbid = row['PDBID']
    if pdbid!=pdbid_prev:
        chains,asmb,covale,modres = pickle.load(gzip.open(f'/projects/ml/RF2_allatom/rcsb/pkl/{pdbid[1:3]}/{pdbid}.pkl.gz'))
        ligands = get_ligands(chains, covale)
    pdbid_prev = pdbid

    # query_ligand = row['LIGAND']
    query_ligand = eval(row['LIGAND'])

    # all assemblies containing this ligand
    ligand_chids = set([res[0] for res in query_ligand])
    ligand_assemblies = [i_a for i_a in asmb 
                         if ligand_chids.issubset(set([x[0] for x in asmb[i_a]]))]

    for i_a in ligand_assemblies:
        asmb_xforms = asmb[i_a]
        asmb_xform_chids = [x[0] for x in asmb_xforms]
        asmb_chains = [chains[i_ch] for i_ch in set(asmb_xform_chids)]

        # assembly must have at least one protein and ligand chain
        if not {'polypeptide(L)','nonpoly'}.issubset(set([ch.type for ch in asmb_chains])):
            continue

        # get query ligand coordinates (one copy of it in this assembly)
        qlig_xyz, qlig_mask, qlig_seq, qlig_chid, qlig_resi, qlig_chxf = \
            get_ligand_xyz(asmb_chains, asmb_xforms, query_ligand)

        if qlig_xyz.numel()==0: continue

        qlig_xyz_valid = qlig_xyz[qlig_mask[:,1],1]
        if qlig_xyz_valid.numel()==0: continue

        # get contacts between this ligand and all transformed protein chains in assembly
        prot_na_contacts = get_contacting_chains(asmb_chains, asmb_xforms, 
                                         qlig_xyz_valid, qlig_chxf)

        lig_contacts = get_contacting_ligands(ligands, asmb_chains, asmb_xforms, 
                                              query_ligand, qlig_xyz_valid, qlig_chxf)

        prot_contacts = [x for x in prot_na_contacts if x[-1]=='polypeptide(L)' and x[2]>0]
        if len(prot_contacts) == 0:
            continue # no protein <5A of ligand, don't use for training

        # pool all contacting objects, sort from most to least contacts (then lowest to highest min distance)
        contacts = sorted(prot_na_contacts+lig_contacts, key=lambda x: (x[2], -x[3]), reverse=True)

        # save results
        new_row = row.copy()
        new_row['ASSEMBLY'] = i_a
        new_row['PROT_CHAIN'] = prot_contacts[0][0] # most-contacting protein chain
        new_row['LIGXF'] = qlig_chxf
        new_row['PARTNERS'] = contacts
        records.append(new_row)

    if i%50 == 0:
        print(i, time.time()-start_time)

In [124]:
df2 = pd.DataFrame.from_records(records)

Put the above in a standalone script to run in parallel on digs.

Generate a list of commands to run

In [6]:
num = 300

In [131]:
with open('cmds_get_partners.txt','w') as outf:
    for istart in range(0, len(df), num):
        # if not os.path.exists(f'results_get_partners/ligands_partners{istart}.csv'):
        print(f'source activate dlchem; python get_partners.py -istart {istart} -num {num}',
              file=outf)

In [7]:
for istart in range(0, len(df), num):
    if not os.path.exists(f'results_get_partners//ligands_partners{istart}.csv'):
        print(istart)

Compile the final list

In [8]:
filenames = [f'results_get_partners/ligands_partners{i}.csv'
             for i in range(0,len(df),num)]

In [9]:
df2 = pd.concat([pd.read_csv(fn, index_col=0, na_filter=False) for fn in filenames])

In [10]:
df2.shape

(385506, 7)

In [29]:
df2.to_csv('ligands_partners.csv',index=None)

## Merge in protein seq clusters

Cross-reference list of multi-residue examples with existing metadata to allow splitting into train & valid sets, and assign protein cluster IDs and ligand cluster sizes, to allow sampling in our current weighted training example sampling framework

In [327]:
df = pd.read_csv('ligands_partners.csv', na_filter=False)

In [328]:
df['CHAINID'] = df.apply(lambda row: str(row['PDBID'])+'_'+row['PROT_CHAIN'], axis=1)

In [329]:
df.shape

(385506, 9)

In [332]:
df[(df['PDBID']=='6j5t') & (df['LIGAND']=="[('P', '501', 'BQL'), ('Q', '502', 'BQO')]")]

,PDBID,LIGAND,COVALENT,ASSEMBLY,PROT_CHAIN,LIGXF,PARTNERS,ID,CHAINID
103702,6j5t,"[('P', '501', 'BQL'), ('Q', '502', 'BQO')]","[(('A', '252', 'VAL', 'C'), ('P', '501', 'BQL'...",1,B,"[('Q', 16)]","[([('P', '501', 'BQL'), ('Q', '502', 'BQO')], ...","6j5t_B_1_[('P', '501', 'BQL'), ('Q', '502', 'B...",6j5t_B


In [326]:
row

CHAINID                                                  6j5t_B
DEPOSITION                                           2019-01-12
RESOLUTION                                                  3.4
HASH                                                      65484
CLUSTER                                                    6397
SEQUENCE      MKKQYLKSGSGTRKEKDKAKRWFLDNGSIFLRELVADCNGKSIPIR...
LEN_EXIST                                                   327
LIGAND                           [(P, 501, BQL), (Q, 502, BQO)]
COVALENT      [((A, 252, VAL, C), (P, 501, BQL, N)), ((A, 25...
ASSEMBLY                                                      1
LIGXF                                                 [(Q, 16)]
PARTNERS      [([('P', '501', 'BQL'), ('Q', '502', 'BQO')], ...
LIGATOMS                                                     53
Name: 252049, dtype: object

In [33]:
df2 = pd.read_csv('/projects/ml/TrRosetta/PDB-2021AUG02/list_v02.csv')

In [34]:
df2.shape

(534579, 7)

In [35]:
df2_subset = df2[['CHAINID','DEPOSITION','RESOLUTION','HASH',
                  'CLUSTER','SEQUENCE','LEN_EXIST']].drop_duplicates('CHAINID')

In [36]:
df3 = df2_subset.merge(df,on='CHAINID')

In [37]:
df3.shape

(362728, 15)

Check which entries get excluded after merging to our old dataset

In [38]:
missing = set(df['CHAINID'].values)-set(df2['CHAINID'].values)

In [39]:
len(missing)

10690

In [287]:
df4 = df[df['CHAINID'].isin(missing)]

In [288]:
df4.to_csv('missing.csv')

The missing entries are newer (2021-) PDBs as well as too-short chains (<60aa). Ivan didn't do sequence clustering on shorter sequences so we will need to leave these out for now.

In [40]:
df3.to_csv('ligands_partners_clustered.csv',index=None)

## Exclude astex / CASF2016 by protein cluster

In [41]:
df = pd.read_csv('ligands_partners_clustered.csv', na_filter=False)

In [283]:
exclude = [line.strip() for line in open('/home/aivan/work/results/2022Jul23/list.astex_diverse').readlines()]

In [284]:
exclude += [line.strip() for line in open('/home/aivan/work/results/2022Jul23/list.astex_nonnat').readlines()]
exclude += [line.strip() for line in open('/home/aivan/work/results/2022Jul23/list.casf2016').readlines()]

In [285]:
len(exclude)

436

In [286]:
exclude = list(set(exclude))

In [287]:
len(exclude)

366

In [288]:
df2 = pd.read_csv('/projects/ml/TrRosetta/PDB-2021AUG02/list_v02.csv')

In [289]:
df2['PDBID'] = df2['CHAINID'].apply(lambda x: x.split('_')[0])

In [290]:
found = np.in1d(np.array(exclude),df2['PDBID'].drop_duplicates().values)

In [291]:
sum(found)

366

In [293]:
df_excl = df2[df2['PDBID'].isin(exclude)]

In [294]:
clus_excl = df_excl['CLUSTER'].drop_duplicates().values

In [53]:
df.shape

(362728, 15)

In [54]:
df_filt = df[~df['CLUSTER'].isin(clus_excl)]

In [55]:
df_filt.shape

(312113, 15)

### Exclude GPCRs

Remove clusters containing GPCRs from the GPCR Dock competitions 2008, 2010, 2013, and 2021

In [56]:
import numpy as np

In [298]:
df2 = pd.read_csv('/projects/ml/TrRosetta/PDB-2021AUG02/list_v02.csv')

In [299]:
gpcrs = ['3eml', '3pbl', '3oeu', '3oe0', '3oe6', '3oe8', '3oe9', '4ib4', '4iar', '4iaq', '4jkv', '4n4w', '4qim', '4qin',
         '7vug', '7vuh', '7vgx']

In [300]:
len(gpcrs)

17

In [301]:
df2['PDBID'] = df2['CHAINID'].apply(lambda x: x.split('_')[0])

In [302]:
found = np.in1d(np.array(gpcrs),df2['PDBID'].drop_duplicates().values)

In [303]:
sum(found)

14

In [304]:
df_excl = df2[df2['PDBID'].isin(gpcrs)]

In [305]:
clus_excl = df_excl['CLUSTER'].drop_duplicates().values

In [306]:
df_filt.shape

(309618, 15)

In [66]:
df_filt = df_filt[~df_filt['CLUSTER'].isin(clus_excl)]

In [67]:
df_filt.shape

(309618, 15)

In [68]:
df_filt.to_csv('sm_compl_all_20230126.csv',index=None)

## Fix covalent entries
The original annotation of covalent bonds isn't correct above. Fix it here.

In [69]:
df = pd.read_csv('sm_compl_all_20230126.csv',na_filter=False)

In [72]:
df['LIGAND'] = df['LIGAND'].apply(lambda x: eval(x))
df['COVALENT'] = df['COVALENT'].apply(lambda x: eval(x))

In [73]:
df['NEWCOVALENT'] = df.apply(lambda row: [bond for bond in row['COVALENT'] 
                                           if (bond[0][:3] in row['LIGAND']) or (bond[1][:3] in row['LIGAND'])],
                              axis=1)

In [74]:
df['COVALENT'] = df['NEWCOVALENT']

In [75]:
df = df.drop(['NEWCOVALENT','PDBID'],axis=1)

In [76]:
df.shape

(309618, 14)

In [77]:
df.to_csv('sm_compl_all_20230126.csv',index=None)

## Mismatched MSAs

### Pilot code

In [70]:
from data_loader import *
from argparse import Namespace

In [71]:
import time

In [78]:
df = pd.read_csv('sm_compl_all_20230126.csv', na_filter=False)

In [79]:
df.shape

(309618, 14)

In [81]:
df['PDBID'] = df['CHAINID'].apply(lambda x: x.split('_')[0])

In [82]:
df['PARTNERS'] = df['PARTNERS'].apply(lambda x: eval(x))

In [83]:
df2 = pd.read_csv('/projects/ml/TrRosetta/PDB-2021AUG02/list_v00.csv')
df2['HASH'] = df2['HASH'].apply(lambda x: f'{x:06d}')

In [84]:
chid2hash = dict(zip(df2['CHAINID'],df2['HASH']))

In [85]:
records = []
for i,row in df.iterrows():
    for p in row['PARTNERS']:
        if p[-1] != 'polypeptide(L)': continue
        
        newrow = row[['CHAINID','LIGAND','ASSEMBLY']].copy()
        newrow['CHAINID2'] = row['PDBID']+'_'+p[0]
        if newrow['CHAINID2'] in chid2hash:
            newrow['HASH2'] = chid2hash[newrow['CHAINID2']]
        records.append(newrow)

In [86]:
df3 = pd.DataFrame.from_records(records)

In [87]:
df3.shape

(769085, 5)

In [88]:
df3.to_csv('sm_compl_all_chainid_expanded.csv')

In [82]:
df = pd.read_csv('sm_compl_all_chainid_expanded.csv')

In [83]:
df['PDBID'] = df['CHAINID'].apply(lambda x: x.split('_')[0])

In [84]:
df['HASH2'] = df['HASH2'].apply(lambda x: f'{int(x):06d}' if pd.notnull(x) else np.nan)

In [85]:
df.shape

(829288, 7)

In [86]:
df = df.drop_duplicates(['PDBID','CHAINID2','HASH2'])

In [87]:
df.shape

(176025, 7)

In [88]:
pdbids = df['PDBID'].drop_duplicates()

In [89]:
len(pdbids)

67910

In [90]:
from data_loader import *

In [91]:
params = set_data_loader_params(Namespace())

In [173]:
t0 = time.time()
ct = 0
records = []
for pdb_id in pdbids[:50]:
    pdb_fn = params['MOL_DIR']+f'/{pdb_id[1:3]}/{pdb_id}.pkl.gz'
    chains, asmb, covale, modres = pickle.loads(gzip.open(pdb_fn.strip(), "rb").read())
    
    tmp = df[df['PDBID']==pdb_id]
    for i,item in tmp.iterrows():
        pdb_chain, pdb_hash = item['CHAINID2'], item['HASH2']
        pdb_id, i_ch_prot = pdb_chain.split('_')

        # transform doesn't actually matter but we need it for featurizing coords
        i_a = str(item['ASSEMBLY'])
        asmb_xfs = asmb[i_a]
        for ch_xf in asmb_xfs:
            if ch_xf[0] == i_ch_prot:
                break

        # load coords
        ch = chains[i_ch_prot]
        xyz_prot, mask_prot, seq_prot, chid_prot, resi_prot = cif_prot_to_xyz(ch, ch_xf, modres)
        protein_L, nprotatoms, _ = xyz_prot.shape
        
        # load msa
        if type(pdb_hash) is not str and np.isnan(pdb_hash):
            item['PROT_LEN'] = 0
        else:
            a3mA = get_msa(params['PDB_DIR'] + '/a3m/'+pdb_hash[:3] + '/'+ pdb_hash + '.a3m.gz', pdb_hash)
            item['PROT_LEN'] = xyz_prot.shape[0]
            item['MATCHED'] = a3mA['msa'].shape[1] == item['PROT_LEN']
        records.append(item)

        ct += 1

        if ct % 50 == 0:
            print(ct, time.time() - t0)

KeyboardInterrupt: 

In [174]:
df2 = pd.DataFrame.from_records(records)

### Run on digs

In [92]:
num = 50

In [93]:
with open('cmds_filter_bad_msas.txt','w') as outf:
    for istart in range(0, len(pdbids), num):
        if not os.path.exists(f'results_filter_msas/bad_msas{istart}.csv'):
            print(f'source activate dlchem; python filter_bad_msas.py -istart {istart} -num {num} ',
                  file=outf)

In [111]:
for istart in range(0, len(pdbids), num):
    if not os.path.exists(f'results_filter_msas/bad_msas{istart}.csv'):
        print(istart)

In [112]:
df2 = pd.concat([pd.read_csv(f'results_filter_msas/bad_msas{istart}.csv')
                 for istart in range(0, len(pdbids), num)])

In [113]:
df2.shape

(176025, 10)

In [114]:
df2['MATCHED'].sum()

173445

In [115]:
df2[df2['PROT_LEN']==0].shape

(2351, 10)

In [117]:
((df2['PROT_LEN']>0) & (df2['MATCHED']==False)).sum()

229

In [118]:
df2.to_csv('bad_msas.csv',index=None)

### Filter datasets

In [8]:
df = pd.read_csv('sm_compl_all_chainid_expanded.csv')

In [93]:
df.shape

(769085, 5)

In [95]:
df['PDBID'] = df['CHAINID'].apply(lambda x: x.split('_')[0])

In [96]:
df2 = pd.read_csv('bad_msas.csv')

In [97]:
df2.shape

(176025, 10)

In [98]:
df2 = df2[df2['MATCHED']==True]

In [99]:
df2.shape

(173445, 10)

In [100]:
matched = set(df2['CHAINID2'].values)

In [101]:
len(matched)

173445

In [102]:
df = pd.read_csv('sm_compl_all_20230126.csv')

In [103]:
df.shape

(309618, 14)

In [104]:
df.drop_duplicates(['CHAINID','LIGAND','ASSEMBLY']).shape

(309618, 14)

In [105]:
df3 = df[df['CHAINID'].isin(matched)]

In [106]:
df3.shape

(309226, 14)

In [107]:
df3['PARTNERS'] = df3['PARTNERS'].apply(lambda x: eval(x))

/scratch/jue/26871179/ipykernel_1179811/2734829755.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['PARTNERS'] = df3['PARTNERS'].apply(lambda x: eval(x))


In [108]:
df3['NUMPARTNERS'] = df3['PARTNERS'].apply(lambda x: len(x))

/scratch/jue/26871179/ipykernel_1179811/447582296.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['NUMPARTNERS'] = df3['PARTNERS'].apply(lambda x: len(x))


In [109]:
df3['PARTNERS'] = df3.apply(
    lambda row: [p for p in row['PARTNERS'] \
                      if p[-1]!='polypeptide(L)' or \
                      (p[-1]=='polypeptide(L)' and (row['CHAINID'].split('_')[0]+'_'+p[0] in matched))],
    axis=1
)

/scratch/jue/26871179/ipykernel_1179811/2549799873.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['PARTNERS'] = df3.apply(


In [110]:
df4 = df3[df3.apply(lambda row: len(row['PARTNERS'])!=row['NUMPARTNERS'], axis=1)]

In [111]:
df4.shape

(4689, 15)

In [118]:
df3 = df3.drop('NUMPARTNERS',axis=1)

In [121]:
df3.to_csv('sm_compl_all_goodmsa_20230126.csv',index=None)

## Filter nonbio, nucleic acids

In [166]:
df = pd.read_csv('sm_compl_all_goodmsa_20230126.csv')

In [167]:
df.shape

(309226, 13)

In [126]:
df['LIGAND'] = df['LIGAND'].apply(lambda x: eval(x))
df['COVALENT'] = df['COVALENT'].apply(lambda x: eval(x))

In [127]:
biolip_list = ['NUC','ZN','CA','MG','III','MN','FE','CU','SF4','FE2','CO','FES','GOL','NA',  
            'CL','K','CU1','GOL','XE','NO2','EDO','NI','BR','CD','O','CS','NO','TL','HG','UNL','KR',
            'SR','RB','F','AG','AR','U','AU','MO','SE','GD','YB','VX','SM','LI','RE','N','W','OS','HO','PI']
LA_list = ['EDO','PG4','OGA','SO4','HEZ','FEO','CL','DMS','ACT','MPD','GOL' ,'NH2','CUA','SIW','PGW','IOD','BR','3NI','ZRW','78M','UNX','nan']
rohith = ['MES','CCN','PO4']
metals = ['LA','NI','3CO','K','CR','ZN','CD','PD','TB','YT3','OS','EU','NA','RB','W','YB','HO3',
          'CE','MN','TL','LI','MN3','AU3','AU','EU3','AL','3NI','FE2','PT','FE','CA','AG','CU1',
          'LU','HG','CO','SR','MG','PB','CS','GA','BA','SM','SB','CU','MO','CU2']

In [128]:
exclude = set(biolip_list+LA_list+rohith)-set(metals)

In [129]:
df['LIGANDNAME'] = df['LIGAND'].apply(lambda x: x[0][2])

Doublecheck that non-biological query ligands are filtered.

In [130]:
df = df[~df['LIGANDNAME'].isin(exclude)]

In [131]:
df.shape

(309226, 14)

In [132]:
df = df.drop('LIGANDNAME',axis=1)

Also remove partner ligands on the exclusion list

In [168]:
df['PARTNERS'] = df['PARTNERS'].apply(lambda x: eval(x))

In [137]:
def func(partners):
    return [
        p for p in partners 
        if (p[-1]!='nonpoly') or \
           ((p[-1]=='nonpoly') and all([lig[2] not in exclude for lig in p[0]]))
    ]       

In [138]:
df['PARTNERS'] = df['PARTNERS'].apply(func)

What about DNA/RNA partners? Are there a lot of these?

In [139]:
df2 = df[df['PARTNERS'].apply(lambda partners: any([(p[-1] != 'nonpoly') and (p[-1] != 'polypeptide(L)')
                                              for p in partners]))]

In [140]:
df2.shape

(13620, 13)

Remove examples where DNA/RNA are the main partner

In [141]:
df.shape

(309226, 13)

In [142]:
df_ = df[df['PARTNERS'].apply(lambda partners: (partners[0][-1]=='polypeptide(L)') or (partners[0][-1]=='nonpoly'))]

In [143]:
df_.shape

(304770, 13)

In [144]:
df = df_

In [145]:
df.to_csv('sm_compl_all_goodmsa_filt_20230126.csv',index=None)

## Count ligand atoms
2023-1-18

Some ligands contain more atoms than the crop size for training. Label examples with # atoms so they can be removed.

Also count number of resolved atoms, to enable filtering out partially resolved ligands

In [58]:
import pickle, gzip, time
from util import cif_ligand_to_xyz, get_ligand_atoms_bonds

In [147]:
df = pd.read_csv('sm_compl_all_goodmsa_filt_20230126.csv')

for col in ['LIGAND','LIGXF','PARTNERS']:
    if col in df:
        df[col] = df[col].apply(lambda x: eval(x))

In [148]:
df.shape

(304770, 13)

In [59]:
records = []

ct = 0
t0 = time.time()
pdbid_prev = None

for i,row in df.iterrows():
    pdbid = row['CHAINID'].split('_')[0]
    if pdbid!=pdbid_prev:
        chains,asmb,covale,modres = pickle.load(gzip.open(f'/projects/ml/RF2_allatom/rcsb/pkl/{pdbid[1:3]}/{pdbid}.pkl.gz'))
    pdbid_prev = pdbid

    # coordinate transforms to recreate this bio-assembly
    i_a = str(row['ASSEMBLY'])
    asmb_xfs = asmb[i_a]

    ligand = row['LIGAND']

    # load query ligand (the "focus ligand" for this training example)
    lig_atoms, lig_bonds = get_ligand_atoms_bonds(ligand, chains, covale)
    lig_ch2xf = dict(row['LIGXF'])

    xyz_sm, mask_sm, msa_sm, chid_sm, lig_akeys = cif_ligand_to_xyz(lig_atoms, asmb_xfs, lig_ch2xf)

    row['LIGATOMS'] = xyz_sm.shape[0]
    row['LIGATOMS_RESOLVED'] = mask_sm.sum().item()
    records.append(row)

    ct += 1
    if ct % 50 == 0:
        print(ct, time.time() -t0)

50 2.4781551361083984


KeyboardInterrupt: 

In [20]:
df2 = pd.DataFrame.from_records(records)

Put the above code into a python script so it can be run in parallel on many nodes to process the full dataset

Generate a list of commands to run

In [159]:
num = 300

In [160]:
with open('cmds_count_atoms.txt','w') as outf:
    for istart in range(0, len(df), num):
        if not os.path.exists(f'results_count_atoms/atom_counts{istart}.csv'):
            print(f'source activate dlchem; python count_atoms.py -istart {istart} -num {num}',
                  file=outf)

In [161]:
for istart in range(0, len(df), num):
    if not os.path.exists(f'results_count_atoms/atom_counts{istart}.csv'):
        print(istart)

Compile the final list

In [162]:
filenames = [f'results_count_atoms/atom_counts{i}.csv'
             for i in range(0,len(df),num)]

In [163]:
df2 = pd.concat([pd.read_csv(fn,index_col=0) for fn in filenames])

In [164]:
df2.shape

(304770, 14)

In [165]:
df2.to_csv('sm_compl_all_goodmsa_filt_atomcounts_20230126.csv',index=None)

In [166]:
df2[df2['LIGATOMS']<=128].shape

(304674, 14)

In [167]:
df2[df2['LIGATOMS']<=192].shape

(304756, 14)

Do we need LIGXF to be a list? i.e. are there ever ligands with residues from different chains?

Yes there are.

In [88]:
df2[df2['LIGAND'].apply(lambda x: len(set([y[0] for y in x]))>1)].shape

(1772, 15)

In [89]:
df2[df2['LIGAND'].apply(lambda x: len(set([y[0] for y in x]))>1)]

,CHAINID,DEPOSITION,RESOLUTION,HASH,CLUSTER,SEQUENCE,LEN_EXIST,LIGAND,COVALENT,ASSEMBLY,PROT_CHAIN,LIGXF,PARTNERS,LIGANDNAME,LIGATOMS
998,4ndz_A,2013-10-28,3.450,37680,9987,KIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEK...,286,"[(G, 1, BDP), (G, 2, GNS), (G, 3, IDR), (G, 4,...",[],1,A,"[(R, 8), (G, 3)]","[([('G', '1', 'BDP'), ('G', '2', 'GNS'), ('G',...",BDP,102
1000,4ndz_B,2013-10-28,3.450,37680,9987,KIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEK...,653,"[(I, 1, BDP), (I, 2, GNS), (I, 3, IDR), (I, 4,...",[],1,B,"[(I, 5), (T, 10)]","[('B', 1, 399, 'polypeptide(L)'), ('C', 2, 96,...",BDP,102
1004,4ndz_D,2013-10-28,3.450,37680,9987,KIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEK...,653,"[(L, 1, BDP), (L, 2, GNS), (L, 3, IDR), (L, 4,...",[],2,D,"[(W, 10), (L, 4)]","[([('L', '1', 'BDP'), ('L', '2', 'GNS'), ('L',...",BDP,102
1007,4ndz_E,2013-10-28,3.450,37680,9987,KIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEK...,653,"[(N, 1, BDP), (N, 2, GNS), (N, 3, IDR), (N, 4,...",[],2,E,"[(Y, 12), (N, 6)]","[([('N', '1', 'BDP'), ('N', '2', 'GNS'), ('N',...",BDP,102
1235,3ne7_A,2010-06-08,2.300,111994,672,XSIEIRKLSIEDLETLIEVARESWKWTYAGIYSEEYIESWIREKYS...,156,"[(C, 161, COA), (D, 162, BME)]",[],1,A,"[(C, 2), (D, 3)]","[('A', 0, 563, 'polypeptide(L)'), ([('B', '160...",COA,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302142,4n8g_D,2013-10-17,1.502,111666,8914,XRILTPRNVLLAAVTTGXXATVSLANAATTLNLSYNGPPDTDKNAV...,303,"[(P, 402, DAL), (Q, 403, DAL)]",[],5,D,"[(P, 2), (Q, 3)]","[('D', 0, 162, 'polypeptide(L)')]",DAL,11
302143,4n8g_D,2013-10-17,1.502,111666,8914,XRILTPRNVLLAAVTTGXXATVSLANAATTLNLSYNGPPDTDKNAV...,303,"[(P, 402, DAL), (Q, 403, DAL)]",[],1,D,"[(P, 15), (Q, 16)]","[('D', 3, 162, 'polypeptide(L)'), ([('M', '402...",DAL,11
302327,1n92_A,2002-11-21,1.470,103064,25154,STAGKVIKCKAAVLWEEKKPFSIEEVEVAPPKAHEVRIKMVATGIC...,374,"[(E, 377, NAJ), (F, 378, PYZ)]",[],1,A,"[(E, 4), (F, 5)]","[('A', 0, 610, 'polypeptide(L)'), ([('C', '375...",NAJ,50
302329,1n92_B,2002-11-21,1.470,103064,25154,STAGKVIKCKAAVLWEEKKPFSIEEVEVAPPKAHEVRIKMVATGIC...,374,"[(J, 377, NAJ), (K, 378, PYZ)]",[],1,B,"[(K, 10), (J, 9)]","[([('J', '377', 'NAJ'), ('K', '378', 'PYZ')], ...",NAJ,50


In [86]:
df2[df2['LIGAND'].apply(lambda x: len(set([y[0] for y in x]))>1)].iloc[0]['LIGAND']

[('G', '1', 'BDP'),
 ('G', '2', 'GNS'),
 ('G', '3', 'IDR'),
 ('G', '4', 'GNS'),
 ('G', '5', 'BDP'),
 ('G', '6', 'NDG'),
 ('G', '7', 'BDP'),
 ('R', '2011', 'NPO')]

## Truncate partner lists
Some partner lists are very long (hundreds of interacting chains/ligands). This will definitely not fit into the eventual crop and wastes compute. Let's truncate partner lists so they contain only the most interacting / closest ligands & protein chains

In [169]:
df = pd.read_csv('sm_compl_all_goodmsa_filt_atomcounts_20230126.csv')

In [170]:
for col in ['LIGAND','LIGXF','COVALENT','PARTNERS']:
    if col in df:
        df[col] = df[col].apply(lambda x: eval(x))

In [171]:
df['NUMLIGPARTNERS'] = df['PARTNERS'].apply(lambda x: len([p for p in x if p[-1]=='nonpoly']))

In [172]:
df['NUMPROTPARTNERS'] = df['PARTNERS'].apply(lambda x: len([p for p in x if p[-1]=='polypeptide(L)']))

In [173]:
df.shape

(304770, 16)

In [174]:
df[df['NUMLIGPARTNERS']<20].shape

(291699, 16)

In [175]:
df[df['NUMPROTPARTNERS']<10].shape

(302609, 16)

After some manual inspection I found a bunch of cases where the partners list is long because it contains many instances of 'DOD', which is deuterated water. These should simply be removed.

In [194]:
df['PARTNERS'] = df['PARTNERS'].apply(lambda x: [p for p in x if not (p[-1]=='nonpoly' and p[0][0][2]=='DOD')])
df['NUMLIGPARTNERS'] = df['PARTNERS'].apply(lambda x: len([p for p in x if p[-1]=='nonpoly']))
df['NUMPROTPARTNERS'] = df['PARTNERS'].apply(lambda x: len([p for p in x if p[-1]=='polypeptide(L)']))

Now let's truncate partner lists so they contain a maximum of 20 ligands and 10 proteins

In [214]:
df[df['NUMLIGPARTNERS']>20].shape

(13051, 16)

In [215]:
df[df['NUMPROTPARTNERS']>10].shape

(2161, 16)

In [220]:
def func(partners):
    newpartners = []
    prot_ct = 0
    prot_num = 10
    lig_ct = 0
    lig_num = 20
    for p in partners:
        if p[-1]=='polypeptide(L)' and prot_ct < prot_num: 
            newpartners.append(p)
            prot_ct+=1
        elif p[-1]=='nonpoly' and lig_ct < lig_num: 
            newpartners.append(p)
            lig_ct+=1
    return newpartners

In [221]:
df['PARTNERS'] = df['PARTNERS'].apply(func)

In [222]:
df['NUMLIGPARTNERS'] = df['PARTNERS'].apply(lambda x: len([p for p in x if p[-1]=='nonpoly']))
df['NUMPROTPARTNERS'] = df['PARTNERS'].apply(lambda x: len([p for p in x if p[-1]=='polypeptide(L)']))

In [225]:
df[df['NUMLIGPARTNERS']>20].shape

(0, 16)

In [226]:
df[df['NUMPROTPARTNERS']>10].shape

(0, 16)

In [229]:
df = df.drop(['PROT_CHAIN','NUMLIGPARTNERS','NUMPROTPARTNERS'],axis=1)

### Remove ligands missing some transforms

Initial attempts to run the block below showed that a small number of ligands have more residues in their `LIGAND` entry than transforms in `LIGXF`, because some of the residues have 0 atom occupancy. On manual inspection these examples are missing atoms even from the ligand residues that exist, so let's filter them out completely.

In [396]:
def func(row):
    ligand = row['LIGAND']
    xf_chs = [xf[0] for xf in row['LIGXF']]
    return any([(res[0] not in xf_chs) for res in ligand])

In [403]:
df = df[~df.apply(func, axis=1)]

There are also some partner ligands that don't have transforms for all the ligand residues.
There are only a few cases, and they all look bad upon manual inspection so we are removing them

In [7]:
npart = df['PARTNERS'].apply(lambda x: len(x))

In [9]:
def func(partners): 
    # remove partners that don't have transforms for all ligand residues
    newpartners = []
    for p in partners:
        if p[-1]=='nonpoly': 
            chxfs = [xf[0] for xf in p[1]]
            if any([res[0] not in chxfs for res in p[0]]):
                continue
        newpartners.append(p)
    return newpartners

In [10]:
df['PARTNERS'] = df['PARTNERS'].apply(func)

In [11]:
npart2 = df['PARTNERS'].apply(lambda x: len(x))

In [12]:
df[npart!=npart2]

,CHAINID,DEPOSITION,RESOLUTION,HASH,CLUSTER,SEQUENCE,LEN_EXIST,LIGAND,COVALENT,ASSEMBLY,LIGXF,PARTNERS,LIGATOMS
39047,1q99_A,2003-08-22,2.11,11245,19102,DYRPGGYHPAFKGEPYKDARYILVRKLGWGHFSTVWLAKDMVNNTH...,354,"[(E, 901, ANP)]",[],1,"[(E, 3)]","[(A, 0, 284, 2.7292721271514893, polypeptide(L...",31
133813,2c5c_G,2005-10-26,2.94,105335,10172,TPDCVTGKVEYTKYNDDDTFTVKVGDKELFTNRWNLQSLLLSAQIT...,69,"[(W, 1, GLA), (W, 2, GLA)]",[],2,"[(W, 5)]","[(G, 1, 118, 2.651259660720825, polypeptide(L)...",23
133814,2c5c_H,2005-10-26,2.94,105335,10172,TPDCVTGKVEYTKYNDDDTFTVKVGDKELFTNRWNLQSLLLSAQIT...,69,"[(X, 1, GLA), (X, 2, GLA)]",[],2,"[(X, 6)]","[(H, 2, 110, 3.026305675506592, polypeptide(L)...",23
133815,2c5c_H,2005-10-26,2.94,105335,10172,TPDCVTGKVEYTKYNDDDTFTVKVGDKELFTNRWNLQSLLLSAQIT...,69,"[(Y, 1, GLC), (Y, 2, GAL), (Y, 3, GLA)]",[],2,"[(Y, 7)]","[(H, 2, 187, 2.400752544403076, polypeptide(L)...",34
133816,2c5c_I,2005-10-26,2.94,105335,10172,TPDCVTGKVEYTKYNDDDTFTVKVGDKELFTNRWNLQSLLLSAQIT...,69,"[(AA, 1, BGC), (AA, 2, GLA), (AA, 3, GLA), (BA...",[],2,"[(KA, 14), (AA, 9), (BA, 10)]","[([('AA', '1', 'BGC'), ('AA', '2', 'GLA'), ('A...",83
133817,2c5c_I,2005-10-26,2.94,105335,10172,TPDCVTGKVEYTKYNDDDTFTVKVGDKELFTNRWNLQSLLLSAQIT...,69,"[(Z, 1, GLA), (Z, 2, GLA)]",[],2,"[(Z, 8)]","[(I, 3, 111, 3.155571222305298, polypeptide(L)...",23
133818,2c5c_J,2005-10-26,2.94,105335,10172,TPDCVTGKVEYTKYNDDDTFTVKVGDKELFTNRWNLQSLLLSAQIT...,69,"[(CA, 1, GLC), (CA, 2, GAL), (CA, 3, GLA), (NA...",[],2,"[(CA, 11), (NA, 17)]","[(J, 4, 118, 2.9804701805114746, polypeptide(L...",49
141133,2z55_C,2007-06-28,2.50,90414,5956,QAGFDLLNDGRPETLWLGIGTLLMLIGTFYFIARGWGVTDKEAREY...,237,"[(P, 270, 22B)]",[],5,"[(P, 12)]","[(C, 0, 185, 3.0629794597625732, polypeptide(L...",54
141134,2z55_C,2007-06-28,2.50,90414,5956,QAGFDLLNDGRPETLWLGIGTLLMLIGTFYFIARGWGVTDKEAREY...,237,"[(P, 270, 22B)]",[],3,"[(P, 4)]","[(C, 0, 185, 3.0629794597625732, polypeptide(L...",54
156702,2cgl_A,2006-03-09,1.88,112674,419,XTFRNCVAVDLGASSGRVXLARYERECRSLTLREIHRFNNGLHSQN...,470,"[(B, 1481, LFR)]",[],1,"[(B, 1)]","[(A, 0, 181, 2.647355079650879, polypeptide(L))]",12


In [15]:
df.to_csv('sm_compl_all_goodmsa_filt_atomcounts_partnersfilt_20230126.csv',index=None)

## Filter covalent ligands

In [246]:
import pickle, gzip, time
from util import cif_ligand_to_xyz, get_ligand_atoms_bonds

In [272]:
def is_weird(covalents):
    """Detects non-biological bonds"""
    has_oxygen_oxygen_bond = any([a1[3][0]=='O' and a2[3][0]=='O' for (a1,a2) in covalents])
    has_fluorine_fluorine_bond = any([a1[3][0]=='F' and a2[3][0]=='F' for (a1,a2) in covalents])
    is_oxy_hydroxy = any([a1[2]=='O' or a2[2]=='O' or \
                          a1[2]=='OH' or a2[2]=='OH' or \
                          a1[2]=='HOH' or a2[2]=='HOH' for (a1,a2) in covalents])
    return has_oxygen_oxygen_bond or has_fluorine_fluorine_bond or is_oxy_hydroxy

In [276]:
def is_clashing(qlig_xyz_valid, xyz_chains, mask_chains, asmb_xforms):
    """Detects if a ligand is within 1A of any protein in its assembly."""
    for xyz, mask in zip(xyz_chains, mask_chains):
        for i_xf, (xf_ch, xf) in enumerate(asmb_xforms):
            if xf_ch != ch.id: continue
            xf = torch.tensor(xf).float()
            u,r = xf[:3,:3], xf[:3,3]
            xyz_xf = torch.einsum('ij,raj->rai', u, xyz) + r[None,None]

            atom_xyz = xyz_xf[:,:NHEAVY][mask[:,:NHEAVY].numpy(),:]
            dist = torch.cdist(qlig_xyz_valid, atom_xyz, compute_mode='donot_use_mm_for_euclid_dist')
            if (dist<1).any():
                return True
    return False

In [389]:
df = pd.read_csv('sm_compl_all_goodmsa_filt_atomcounts_partnersfilt_20230126.csv')

In [390]:
for col in ['LIGAND','LIGXF','COVALENT','PARTNERS']:
    if col in df:
        df[col] = df[col].apply(lambda x: eval(x))

In [391]:
df.shape

(304770, 13)

In [379]:
records = []

ct = 0
t0 = time.time()

pdbid_prev = None
i_a_prev = None
for i,row in tmp.iterrows():
    pdbid = row['CHAINID'].split('_')[0]
    if pdbid!=pdbid_prev:
        chains,asmb,covale,modres = pickle.load(gzip.open(f'/projects/ml/RF2_allatom/rcsb/pkl/{pdbid[1:3]}/{pdbid}.pkl.gz'))

    # remove covalent examples with non-biological bonds
    if len(row['COVALENT'])>0:
        if is_weird(row['COVALENT']):
            print('non-biological protein-ligand bond',row['CHAINID'],row['LIGAND'])
            continue
            
    i_a = str(row['ASSEMBLY'])
    asmb_xforms = asmb[i_a]
    asmb_xform_chids = [x[0] for x in asmb_xforms]
    asmb_chains = [chains[i_ch] for i_ch in set(asmb_xform_chids)]

    if pdbid!=pdbid_prev or i_a!=i_a_prev:
        # featurize all protein chains
        xyz_chains, mask_chains = [], []
        for ch in asmb_chains:
            if ch.type != 'polypeptide(L)': continue
            xyz, mask, seq, chid, resi, unrec_elements = chain_to_xyz(ch)
            xyz_chains.append(xyz)
            mask_chains.append(mask)
        
    pdbid_prev = pdbid
    i_a_prev = i_a
    
    # remove examples with clashes between query ligand and protein chains
    qlig_xyz, qlig_mask, qlig_seq, qlig_chid, qlig_resi, qlig_chxf = \
        get_ligand_xyz(asmb_chains, asmb_xforms, ligand, 
                       seed_ixf=dict(row['LIGXF'])[ligand[0][0]])
    qlig_xyz_valid = qlig_xyz[qlig_mask]
    clash = is_clashing(qlig_xyz_valid, xyz_chains, mask_chains, asmb_xforms)
    if clash:
        print('clash between query ligand and protein',row['CHAINID'], row['LIGAND'])
        continue

    # filter partners
    new_partners = []
    for p in row['PARTNERS']:
        if p[-1]=='nonpoly':
            # remove covalent partners with non-biological bonds
            bonds = []
            for bond in covale:
                if any([bond.a[:3]==res or bond.b[:3]==res for res in p[0]]):
                    bonds.append((bond.a, bond.b))
            if len(bonds)>0:
                if is_weird(bonds):
                    print('non-biological protein-ligand bond in partner',row['CHAINID'],p)
                    continue
            # remove partners with clash to protein
            lig_xyz, lig_mask, lig_seq, lig_chid, lig_resi, lig_chxf = \
                get_ligand_xyz(asmb_chains, asmb_xforms, plig, 
                               seed_ixf=dict(p[1])[plig[0][0]])

            lig_xyz_valid = lig_xyz[lig_mask]
            clash = is_clashing(lig_xyz_valid, xyz_chains, mask_chains, asmb_xforms)
            if clash:
                print('clash between partner ligand and protein',row['CHAINID'],p)
                continue
        new_partners.append(p)

    row['PARTNERS'] = new_partners
    records.append(row)

    ct += 1
    if ct % 50 == 0:
        print(ct, time.time() -t0)

    print(ct, time.time() -t0)

1 1.690089464187622
2 1.695704460144043
3 3.4056851863861084
4 3.4112119674682617
5 3.4163084030151367
6 3.426068067550659
7 3.4309725761413574
8 3.4358325004577637
9 3.440671682357788
10 3.4455316066741943
11 3.450380802154541
12 3.4552924633026123
13 3.460136651992798
14 3.5645902156829834
15 3.8599674701690674
16 3.862327814102173
17 4.0322442054748535
18 4.034143686294556
19 4.237951755523682
20 4.443865060806274
21 4.446354150772095
22 4.653178691864014
23 4.833681344985962
24 4.835554122924805
25 4.837449550628662
26 4.839299201965332
27 4.921200513839722
28 6.200256109237671
29 6.222212076187134
30 6.24448561668396
31 6.265885353088379
32 6.292298793792725
33 6.313573837280273
34 6.339301109313965
35 6.364566087722778
36 6.38495945930481
37 6.411026477813721
38 12.637141227722168
39 12.663028478622437
40 12.688291549682617
41 12.713476419448853
42 12.747552394866943
43 12.772982120513916
44 12.803429365158081
45 12.83358883857727
46 12.858815908432007
47 12.889292001724243
48 13

In [380]:
df2 = pd.DataFrame.from_records(records)

Put the above code into a python script so it can be run in parallel on many nodes to process the full dataset

Generate a list of commands to run

In [235]:
num = 300

In [282]:
with open('cmds_filter_covalents.txt','w') as outf:
    for istart in range(0, len(df), num):
        if not os.path.exists(f'results_filter_covalents/filtered_covalents{istart}.csv'):
            # df.iloc[istart:istart+num].to_csv(f'input{istart}.csv')
            print(f'source activate dlchem; python filter_covalents.py -istart {istart} -num {num}',
                  file=outf)

In [385]:
for istart in range(0, len(df), num):
    if not os.path.exists(f'results_filter_covalents/filtered_covalents{istart}.csv'):
        print(istart)

Compile the final list

In [386]:
filenames = [f'results_filter_covalents/filtered_covalents{i}.csv'
             for i in range(0,len(df),num)]

In [387]:
df2 = pd.concat([pd.read_csv(fn,index_col=0) for fn in filenames])

In [388]:
df2.shape

(304474, 14)

In [16]:
df2.to_csv('sm_compl_all_covfilt_20230126.csv',index=None)

NameError: name 'df2' is not defined

## Filter on contacts, remove nucleic acid examples

In [217]:
df = pd.read_csv('sm_compl_all_covfilt_20230126.csv')

In [218]:
for col in ['LIGAND','LIGXF','COVALENT','PARTNERS']:
    if col in df:
        df[col] = df[col].apply(lambda x: eval(x))

Remove ligands that don't make many contacts with proteins. Tweaked the filter below a few times after manual inspection to arrive at a conservative criterion for ligands "floating in space"

In [219]:
idx = df.apply(lambda row: 
                   row['LIGAND'][0][2] in metals or \
                   sum([p[2] for p in row['PARTNERS']])>10 or \
                   len(row['COVALENT'])>0,
                   axis=1)

In [220]:
# see what would be filtered out
df2 = df[~idx]

In [221]:
df2.shape

(230, 13)

In [222]:
# apply the filter
df = df[idx]

In [223]:
df.shape

(304236, 13)

Remove examples with DNA/RNA partners. Originally I planned to just ignore these partners but they make up pretty large portions of certain assemblies. We should wait until we can featurize these properly, and then add them back in.

Since I already removed nucleic acid partners, I need to use a previous list to identify these examples, and then remove them from the current list

In [182]:
df3 = pd.read_csv('sm_compl_all_goodmsa_20230126.csv')

In [183]:
df3.shape

(309226, 13)

In [184]:
df3['PARTNERS'] = df3['PARTNERS'].apply(lambda x: eval(x))

In [185]:
idx = df3['PARTNERS'].apply(
    lambda partners: any([p[-1]!='nonpoly' and p[-1]!='polypeptide(L)' for p in partners]))

In [186]:
idx.sum()

13620

In [226]:
df['LIGAND'] = df['LIGAND'].astype(str)

In [190]:
df.drop_duplicates(['CHAINID','LIGAND','ASSEMBLY']).shape

(304236, 13)

In [191]:
to_remove = df3[idx]

In [192]:
to_remove['LIGAND'] = to_remove['LIGAND'].astype(str)

/scratch/jue/27013319/ipykernel_472328/3300671840.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  to_remove['LIGAND'] = to_remove['LIGAND'].astype(str)


In [193]:
to_remove['to_remove']=True

/scratch/jue/27013319/ipykernel_472328/736268148.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  to_remove['to_remove']=True


In [227]:
df_ = df.merge(to_remove[['CHAINID','LIGAND','ASSEMBLY','to_remove']], on=['CHAINID','LIGAND','ASSEMBLY'], how='left')

In [230]:
df_.shape

(304236, 14)

In [231]:
(df_['to_remove']==True).sum()

9154

In [232]:
df = df_[df_['to_remove']!=True]

In [233]:
df.shape

(295082, 14)

In [234]:
df.to_csv('sm_compl_all_finalfilt_20230127.csv',index=None)

## Split into different datasets

In [74]:
df = pd.read_csv('sm_compl_all_finalfilt2_20230201.csv')

In [75]:
for col in ['LIGAND','LIGXF','COVALENT','PARTNERS']:
    if col in df:
        df[col] = df[col].apply(lambda x: eval(x))

In [76]:
df.shape

(295082, 14)

For some reason there are entries with duplicated covalent bonds. deduplicate these.

In [77]:
tmp = df[df['COVALENT'].apply(lambda x: len(x)!=len(set(x)))]

In [78]:
tmp.shape

(213, 14)

In [79]:
def func(covalent):
    new_covalent = []
    for bond in covalent:
        if bond not in new_covalent:
            new_covalent.append(bond)
    return new_covalent

In [80]:
df['COVALENT'] = df['COVALENT'].apply(func)

In [81]:
tmp = df[df['COVALENT'].apply(lambda x: len(x)!=len(set(x)))]

In [82]:
tmp.shape

(0, 14)

In [83]:
df.shape

(295082, 14)

### Assemblies
After some exploratory analysis I chose the following definition of assembly. These examples will be in their own dataset that will always attempt to featurize every single ligand in the partner list.

The remaining examples will be put in datasets that will only attempt to featurize the query ligand and its main interacting protein chain.

In [84]:
idx = df['PARTNERS'].apply(
    lambda partners: 
        len([p for p in partners if p[-1]=='nonpoly' and p[2]>0]) + \
        len([p for p in partners if p[-1]=='polypeptide(L)' and p[2]>10]) > 1)

In [85]:
df_asmb = df[idx]

In [86]:
df_asmb.shape

(153091, 14)

In [87]:
df_asmb.to_csv('sm_compl_asmb_20230201.csv',index=None)

In [88]:
df = df[~idx]

### Multi-residue

In [89]:
idx = df['LIGAND'].apply(lambda x: len(x)>1) & df['COVALENT'].apply(lambda x: len(x)==0)

In [90]:
df_multi = df[idx]

In [91]:
df_multi.shape

(3303, 14)

In [92]:
df_multi.to_csv('sm_compl_multi_20230201.csv',index=None)

In [93]:
df = df[~idx]

### Covalent

In [94]:
idx = df['COVALENT'].apply(lambda x: len(x)>0)

In [95]:
df_cov = df[idx]

In [96]:
df_cov.shape

(8606, 14)

In [97]:
df_cov.to_csv('sm_compl_covalent_20230201.csv',index=None)

In [98]:
df = df[~idx]

### Metal ions

In [100]:
df['LIGANDNAME'] = df['LIGAND'].apply(lambda x: x[0][2])

In [101]:
metals = ['LA','NI','3CO','K','CR','ZN','CD','PD','TB','YT3','OS','EU','NA','RB','W','YB','HO3',
          'CE','MN','TL','LI','MN3','AU3','AU','EU3','AL','3NI','FE2','PT','FE','CA','AG','CU1',
          'LU','HG','CO','SR','MG','PB','CS','GA','BA','SM','SB','CU','MO','CU2']

In [102]:
idx = df['LIGANDNAME'].isin(metals)

In [103]:
df_metal = df[idx]

In [104]:
df_metal.shape

(74198, 15)

In [105]:
df_metal = df_metal.drop(['LIGANDNAME'],axis=1)

In [106]:
df_metal.to_csv('metal_compl_20230201.csv',index=None)

In [107]:
df = df[~idx]

In [108]:
df = df.drop(['LIGANDNAME'],axis=1)

### Organic ligands
What is left at this point is just the "basic" ligand-protein complexes

In [109]:
df.shape

(55884, 14)

In [110]:
df.to_csv('sm_compl_20230201.csv',index=None)

## Strict validation set
2023-1-16

Re-create the strict validation set with no tanimoto overlap with train

### Functions

In [111]:
import pickle, gzip, time
from openbabel import openbabel
sys.path.insert(0,'/home/jue/git/rf2a-fd3/')
import chemical
from chemical import atom_num, frame_priority2atom

In [97]:
def cif_ligand_to_xyz(atoms, asmb_xfs=None, ch2xf=None):

    elnum_to_atom = dict(zip(atom_num, frame_priority2atom))
    atoms_no_H = {k:v for k,v in atoms.items() if v.element != 1} # exclude hydrogens
    L = len(atoms_no_H)

    xyz = torch.zeros(L, 3)
    mask = torch.zeros(L,).bool()
    seq = torch.full((L,), np.nan)
    chid = ['-']*L
    akeys = [None]*L
    
    # create coords, atom mask, and seq tokens
    for i,(k,v) in enumerate(atoms_no_H.items()):
        xyz[i, :] = torch.tensor(v.xyz)
        mask[i] = v.occ
        if v.element not in elnum_to_atom:
            print('Element not in alphabet:',v.element)
            seq[i] = chemical.aa2num['ATM']
        else:
            seq[i] = chemical.aa2num[elnum_to_atom[v.element]]
        akeys[i] = k
        chid[i] = k[0]
        
    if asmb_xfs is not None:
        # apply transforms
        chid = np.array(chid)
        for i_ch in np.unique(chid):
            idx = chid==i_ch
            xf = torch.tensor(asmb_xfs[ch2xf[i_ch]][1]).float()
            u,r = xf[:3,:3], xf[:3,3]
            xyz[idx] = torch.einsum('ij,aj->ai', u, xyz[idx]) + r[None,None]
        
    return xyz, mask, seq, chid, akeys

def cif_ligand_to_obmol(xyz, akeys, atoms, bonds):
    
    mol = openbabel.OBMol()    
    for i,k in enumerate(akeys):
        a = mol.NewAtom()
        a.SetAtomicNum(atoms[k].element)
        a.SetVector(float(xyz[i,0]), float(xyz[i,1]), float(xyz[i,2]))

    sm_L = len(akeys)
    bond_feats = torch.zeros((sm_L,sm_L))
    for bond in bonds:
        if bond.a not in akeys or bond.b not in akeys: continue # intended to skip bonds to H's
        i = akeys.index(bond.a)
        j = akeys.index(bond.b)
        bond_feats[i,j] = bond.order if not bond.aromatic else 4
        bond_feats[j,i] = bond_feats[i,j]

        obb = openbabel.OBBond()
        obb.SetBegin(mol.GetAtom(i+1))
        obb.SetEnd(mol.GetAtom(j+1))
        obb.SetBondOrder(bond.order)
        if bond.aromatic:
            obb.SetAromatic()

        mol.AddBond(obb)
    
    return mol, bond_feats

### Get ligand names

In [279]:
df = pd.read_csv('sm_compl_all_goodmsa_20230116.csv')

for col in ['LIGAND','PARTNERS','LIGXF']:
    df[col] = df[col].apply(lambda x: eval(x))

df['LIGNAME'] = df['LIGAND'].apply(lambda x: x[0][2])

In [280]:
df2 = df.drop_duplicates('LIGNAME')

In [281]:
df2.shape

(17468, 14)

In [285]:
df2 = df2[~df2['LIGNAME'].isin(metals)]

In [286]:
df2.shape

(17424, 14)

Also need to compare to ligands in old dataset, since we've been training fold & dock 3 on it

In [314]:
df_old = pd.read_csv('/projects/ml/RF2_allatom/list_v02_sm_filt_notest.csv',index_col=None,dtype=str)

In [315]:
df_old['LIGANDS'] = df_old['LIGANDS'].apply(lambda x: eval(x))

In [316]:
records = []
for i,row in df_old.iterrows():
    for lig in row['LIGANDS']:
        newrow = row.copy()
        newrow['LIGNAME'] = lig.split('_')[1]
        records.append(newrow)

In [317]:
df3 = pd.DataFrame.from_records(records)

In [318]:
df3.shape

(74996, 10)

In [319]:
df3 = df3.drop_duplicates('LIGNAME')

In [320]:
df3.shape

(18229, 10)

In [321]:
df3.to_csv('old_ligands.csv',index=False)

In [322]:
new = set(df3['LIGNAME'].values)-set(df2['LIGNAME'].values)

In [323]:
len(new)

3438

In [324]:
lignames = set(df3['LIGNAME'].values).union(set(df2['LIGNAME'].values))

In [327]:
len(lignames)

20862

In [328]:
df3 = pd.read_csv('ligands.csv')

In [329]:
df3['LIGAND'] = df3['LIGAND'].apply(lambda x: eval(x))

In [330]:
df3['LIGNAME'] = df3['LIGAND'].apply(lambda x: list(x)[0][2])

In [331]:
df3 = df3[df3['LIGNAME'].isin(lignames)].drop_duplicates('LIGNAME')

In [332]:
df3.shape

(20761, 4)

In [334]:
df3.to_csv('ligands_for_tanimoto.csv')

### Copy mol2's of ligands only in old set

About 100 are missing. Are these in the old set?

In [308]:
missing = set(lignames)-set(df3['LIGNAME'])

In [333]:
len(missing)

101

In [335]:
df = pd.read_csv('old_ligands.csv')

In [337]:
df = df[df['LIGNAME'].isin(missing)]

In [338]:
df.shape

(101, 10)

In [343]:
df['LIGANDS'] = df['LIGANDS'].apply(lambda x: eval(x))

Yes, so we'll have to copy over their mol2 files separately.

In [340]:
import shutil

In [364]:
for i,row in df.iterrows():
    for lig in row['LIGANDS']:
        if row['LIGNAME'] in lig:
            break
            
    filenames = glob.glob('/projects/ml/RF2_allatom/by-pdb/'+
        row['CHAINID'][1:3]+'/'+row['CHAINID'][:4]+'_'+row['LIGNAME']+'*.mol2')

    outfn = row['LIGNAME']+'.mol2'

    outdir = 'mol2/'+outfn[0]+'/'
    os.makedirs(outdir, exist_ok=True)

    shutil.copyfile(filenames[0], outdir+outfn)

### Save ligands from cif to mol2

In [365]:
df = pd.read_csv('ligands_for_tanimoto.csv')

In [366]:
df.shape

(20761, 5)

In [370]:
df['LIGAND'] = df['LIGAND'].apply(lambda x: eval(x))

In [371]:
ct = 0
t0 = time.time()
for i,row in df.iterrows():

    pdb_id = row['PDBID']
    ligand = list(row['LIGAND'])

    chains, asmb, covale, modres = pickle.load(gzip.open(f'/projects/ml/RF2_allatom/rcsb/pkl/{pdb_id[1:3]}/{pdb_id}.pkl.gz'))

    lig_atoms = dict()
    lig_bonds = []
    for i_ch,ch in chains.items():
        for k,v in ch.atoms.items():
            if k[:3] in ligand:
                lig_atoms[k] = v
        for bond in ch.bonds:
            if bond.a[:3] in ligand or bond.b[:3] in ligand:
                lig_bonds.append(bond)
    # for bond in covale:
    #     if bond.a[:3] in ligand and bond.b[:3] in ligand:
    #         lig_bonds.append(bond)

    xyz_sm, mask_sm, msa_sm, chid_sm, akeys = cif_ligand_to_xyz(lig_atoms)

    mol, bond_feats_sm = cif_ligand_to_obmol(xyz_sm, akeys, lig_atoms, lig_bonds)

    obConversion = openbabel.OBConversion()
    obConversion.SetOutFormat("mol2")
    outfn = ligand[0][2]+'.mol2'
    outdir = f'mol2/{outfn[0]}/'
    os.makedirs(outdir, exist_ok=True)
    obConversion.WriteFile(mol, outdir+outfn)
    
    ct += 1
    
    if ct % 50 == 0:
        print(ct, time.time() - t0, ligand)

KeyboardInterrupt: 

Put the above in a standalone script to run in parallel on digs.

Generate a list of commands to run

In [372]:
df.shape

(20761, 5)

In [373]:
num = 100

In [374]:
with open('cmds_lig_to_mol2.txt','w') as outf:
    for istart in range(0, len(df), num):
        print(f'source activate dlchem; python lig_to_mol2.py -istart {istart} -num {num}',
              file=outf)

### Compute tanimoto

In [375]:
from openbabel import openbabel, pybel
openbabel.obErrorLog.SetOutputLevel(0) # disable openbabel warnings

In [376]:
filenames = glob.glob('mol2/*/*.mol2')

In [377]:
len(filenames)

20862

In [378]:
t0 = time.time()
mols = [next(pybel.readfile('mol2',fn)) for fn in filenames]
fps = [mol.calcfp() for mol in mols]

In [379]:
sim_s = []
ct = 0
for fp in fps:
    sim = np.array([fp|fp2 for fp2 in fps])
    sim_s.append(sim)
    ct += 1
    if ct%100 == 0:
        print(time.time()-t0)

200.96962189674377
202.82005214691162
204.62919449806213
206.47700548171997
208.45842242240906
210.2435073852539
212.09637117385864
214.01284098625183
215.9991946220398
217.95119738578796
219.68930292129517
221.39617609977722
223.13687133789062
224.90238904953003
226.68455338478088
228.3933811187744
230.19573545455933
232.1125407218933
233.9922971725464
235.85839891433716
237.74592900276184
239.45363926887512
241.20519161224365
242.99975514411926
244.72403478622437
246.4277424812317
248.19553351402283
250.0407178401947
251.89701128005981
253.65857601165771
255.48559498786926
257.20835185050964
258.96954584121704
260.7551484107971
262.45608019828796
264.16799783706665
265.96162700653076
267.79862570762634
269.6092894077301
271.3825283050537
273.1717450618744
274.92721486091614
276.6996970176697
278.4632840156555
280.1786003112793
281.9093804359436
283.71502780914307
285.5757722854614
287.3399143218994
289.14551067352295
290.870304107666
292.6472382545471
294.4373996257782
296.1445569992

In [380]:
sim_s[0].shape

(20862,)

In [381]:
sim = np.stack(sim_s, axis=0)

In [382]:
sim.shape

(20862, 20862)

In [383]:
names = [os.path.basename(fn).replace('.mol2','') for fn in filenames]

In [384]:
np.savez('tanimoto_ligands.npz',names=names,sim=sim)

### Curate strict valid set

In [117]:
val_pdb_ids = set([int(l) for l in open('/projects/ml/RF2_allatom/valid_remapped').readlines()])

In [118]:
# df = pd.read_csv('sm_compl_20230118.csv') # previously used this larger list

In [119]:
# get the "train" split from these sets
df = pd.concat([
    pd.read_csv('sm_compl_20230201.csv'),
    pd.read_csv('sm_compl_asmb_20230201.csv'),
    pd.read_csv('sm_compl_multi_20230201.csv'),
    pd.read_csv('sm_compl_covalent_20230201.csv') 
])
for col in ['LIGAND','PARTNERS','LIGXF']:
    df[col] = df[col].apply(lambda x: eval(x))

In [120]:
df_train = df[~df['CLUSTER'].isin(val_pdb_ids)]

In [121]:
df_train.shape

(198686, 14)

In [122]:
def func(row):
    lignames = [lig[2] for lig in row['LIGAND']]
    for p in row['PARTNERS']:
        if p[-1]=='nonpoly':
            lignames += [lig[2] for lig in p[0]]
    return lignames

In [123]:
df_train['LIGNAMES'] = df_train.apply(func, axis=1)

/scratch/jue/27456429/ipykernel_429561/3330530388.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['LIGNAMES'] = df_train.apply(func, axis=1)


In [124]:
train_lignames = set()
for i,row in df_train.iterrows():
    train_lignames.update(row['LIGNAMES'])

In [125]:
len(train_lignames)

17012

In [126]:
# get the "valid" split from these sets
df2 = pd.read_csv('sm_compl_20230201.csv') 
for col in ['LIGAND','PARTNERS','LIGXF']:
    df2[col] = df2[col].apply(lambda x: eval(x))
df2['LIGNAME'] = df2['LIGAND'].apply(lambda x: x[0][2])

In [127]:
df_valid = df2[df2['CLUSTER'].isin(val_pdb_ids)]

In [128]:
df_valid.shape

(3280, 15)

In [129]:
dat = np.load('tanimoto_ligands.npz')

In [130]:
names = dat['names']

In [131]:
len(names)

20862

In [132]:
sim = dat['sim']

In [133]:
sim.shape

(20862, 20862)

In [134]:
df = pd.read_csv('old_ligands.csv')

In [135]:
df_train_old = df[~df['CLUSTER'].isin(val_pdb_ids)]

In [136]:
train_lignames = train_lignames.union(set(df_train_old['LIGNAME'].values))
name_in_train = np.array([name in train_lignames for name in names])

In [137]:
len(name_in_train)

20862

In [138]:
sum(name_in_train)

19760

In [139]:
name2idx = dict(zip(names,range(len(names))))

In [140]:
def func(ligname):
    return name_in_train[sim[name2idx[ligname]]>0.85].any()

In [141]:
df_valid['TRAIN_OVERLAP'] = df_valid['LIGNAME'].apply(func)

/scratch/jue/27456429/ipykernel_429561/3353789439.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_valid['TRAIN_OVERLAP'] = df_valid['LIGNAME'].apply(func)


In [142]:
df_strict = df_valid[~df_valid['TRAIN_OVERLAP']]

In [143]:
df_strict.shape

(662, 16)

In [144]:
df_strict['CLUSTER'].drop_duplicates().shape

(61,)

In [145]:
df_strict = df_strict.drop(['LIGNAME','TRAIN_OVERLAP'],axis=1)

In [146]:
df_strict.to_csv('sm_compl_valid_strict_20230201.csv',index=None)